# 02 — Quality Validation

Technical validation checks for the **Scientific Data** Data Descriptor.

**Manuscript section:** Technical Validation

**Outputs:** validation tables saved under `notebooks/data_paper/figures/`.

In [ ]:
from pathlib import Path

import polars as pl
from deltalake import DeltaTable

from config.paths import (
    BRONZE_COLETAS,
    BRONZE_FACE,
    REPO_ROOT,
    SILVER_FACE_CLEAN,
    SILVER_PROCESSOS,
)

FIGURES_DIR = REPO_ROOT / "projects/litigancia/notebooks/data_paper/figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

lf_processos = pl.scan_delta(str(SILVER_PROCESSOS))
lf_face = pl.scan_delta(str(SILVER_FACE_CLEAN))


## 1. Null and completeness rates (`processos_delta`)

In [ ]:
KEY_COLS = [
    "id_processo",
    "cd_processo",
    "classe",
    "assunto",
    "comarca",
    "foro",
    "vara",
    "magistrado",
    "data_disponibilizacao",
    "decisao",
    "source_bronze_path",
]

n_total = lf_processos.select(pl.len()).collect().item()

nulls = lf_processos.select(
    [pl.col(c).null_count().alias(c) for c in KEY_COLS]
).collect()

null_rates = nulls.transpose(include_header=True, header_name="column", column_names=["null_count"]).with_columns(
    (pl.col("null_count") / n_total * 100).round(4).alias("null_pct")
)

print(f"Total rows: {n_total:,}")
display(null_rates)
null_rates.write_csv(FIGURES_DIR / "table_null_rates_processos.csv")

## 2. Date parsing and temporal gaps

In [ ]:
date_check = lf_processos.with_columns(
    pl.col("data_disponibilizacao")
    .str.strptime(pl.Date, "%d/%m/%Y", strict=False)
    .alias("dt_pub")
).select(
    pl.len().alias("n_total"),
    pl.col("dt_pub").null_count().alias("n_unparseable_dates"),
    pl.col("dt_pub").min().alias("min_date"),
    pl.col("dt_pub").max().alias("max_date"),
).collect()

display(date_check)
date_check.write_csv(FIGURES_DIR / "table_date_parsing.csv")

## 3. Duplicate key check

In [ ]:
dup_check = (
    lf_processos.group_by("id_processo", "decisao")
    .agg(pl.len().alias("n"))
    .filter(pl.col("n") > 1)
    .select(pl.len().alias("n_duplicate_keys"), pl.col("n").sum().alias("n_extra_rows"))
    .collect()
)

if dup_check.is_empty():
    dup_summary = pl.DataFrame({"n_duplicate_keys": [0], "n_extra_rows": [0]})
else:
    dup_summary = dup_check

display(dup_summary)
dup_summary.write_csv(FIGURES_DIR / "table_duplicate_keys.csv")

## 4. Cross-table join coverage

In [ ]:
cds_processos = lf_processos.select(pl.col("cd_processo").unique().alias("cd_processo"))
cds_face = lf_face.select(pl.col("cd_processo").unique().alias("cd_processo"))

join_coverage = pl.concat(
    [
        pl.DataFrame({"metric": ["processos_unique_cd"], "value": [cds_processos.collect().height]}),
        pl.DataFrame({"metric": ["face_unique_cd"], "value": [cds_face.collect().height]}),
    ]
)

overlap = (
    cds_processos.join(cds_face, on="cd_processo", how="inner")
    .select(pl.len().alias("n_overlap"))
    .collect()
)

join_coverage = pl.concat(
    [
        join_coverage,
        pl.DataFrame({"metric": ["overlap_cd"], "value": [overlap.item()]}),
    ]
)

display(join_coverage)
join_coverage.write_csv(FIGURES_DIR / "table_join_coverage.csv")

## 5. FACE value sanity checks

In [ ]:
face_anomalies = lf_face.select(
    pl.len().alias("n_total"),
    (pl.col("valor_corrigido_atual") < 0).sum().alias("n_valor_negativo"),
    (pl.col("tempo_tramitacao_meses") < 0).sum().alias("n_tramitacao_negativa"),
    (pl.col("valor_corrigido_atual") > 1_000_000_000).sum().alias("n_valor_acima_1bi"),
).collect()

display(face_anomalies)
face_anomalies.write_csv(FIGURES_DIR / "table_face_anomalies.csv")

## 6. Bronze layer row counts (pipeline consistency)

In [ ]:
def delta_row_count(path: Path) -> int | None:
    if not path.exists():
        return None
    if not DeltaTable.is_deltatable(str(path)):
        return None
    return pl.scan_delta(str(path)).select(pl.len()).collect().item()

layer_counts = pl.DataFrame(
    {
        "layer": [
            "bronze_coletas",
            "silver_processos",
            "bronze_face",
            "silver_face_clean",
        ],
        "path": [
            str(BRONZE_COLETAS),
            str(SILVER_PROCESSOS),
            str(BRONZE_FACE),
            str(SILVER_FACE_CLEAN),
        ],
        "row_count": [
            delta_row_count(BRONZE_COLETAS),
            delta_row_count(SILVER_PROCESSOS),
            delta_row_count(BRONZE_FACE),
            delta_row_count(SILVER_FACE_CLEAN),
        ],
    }
)

display(layer_counts)
layer_counts.write_csv(FIGURES_DIR / "table_layer_row_counts.csv")